In [47]:
import torch
from torch import nn

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
).to(device)

In [48]:
x_2 = inputs[1]
d_in = inputs.shape[1]

d_out = 2

In [49]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = False).to(device)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = False).to(device)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = False).to(device)

In [50]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

query_2

tensor([0.4306, 1.4551], device='cuda:0')

In [51]:
key_2

tensor([0.4433, 1.1419], device='cuda:0')

In [52]:
value_2

tensor([0.3951, 1.0037], device='cuda:0')

In [53]:
keys = inputs  @ W_key
queries = inputs  @ W_query
values = inputs  @ W_value

In [54]:
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], device='cuda:0')

In [55]:
queries

tensor([[0.2309, 1.0966],
        [0.4306, 1.4551],
        [0.4300, 1.4343],
        [0.2355, 0.7990],
        [0.2983, 0.6565],
        [0.2568, 1.0533]], device='cuda:0')

In [56]:
values

tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]], device='cuda:0')

In [57]:
# Check the relation between each queries and keys, how much they relate to each other using dot product (cosing)
query_2 = queries[1]
attn_scores_2 = query_2 @ keys.T

attn_scores_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440], device='cuda:0')

In [58]:
attn_scores = queries @ keys.T
attn_scores

tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]], device='cuda:0')

In [59]:
# Attention weights (normalized attention scores)
# Scale by sqrt(d_keys)
d_k = keys.shape[-1]
attn_weights = torch.softmax(attn_scores / (d_k ** 0.5), dim = -1)
print(d_k)
attn_weights

2


tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]], device='cuda:0')

In [60]:
torch.cuda.is_available()

True

In [61]:
context_vector = attn_weights @ values
context_vector

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], device='cuda:0')

In [68]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out, device = torch.device("cuda"), qkv_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias).to(device)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias).to(device)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias).to(device)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_score = queries @ keys.T
        attn_weights = torch.softmax(
            attn_score / (keys.shape[-1] ** 0.5), dim = -1
        )

        context_vector = attn_weights @ values

        return context_vector

In [70]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v1(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], device='cuda:0', grad_fn=<MmBackward0>)
